# Conditions météorologiques et régularité des TGV en France

## 1. Introduction
(à rédiger)

### Chargement des bibliothèques et des fonctions

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# Récupération des données
from chargement_sncf import charger_sncf
from chargement_gares import charger_gares

# Préparation des données SNCF
from preparation_sncf import (
    lister_gares,
    filtrer_liaisons_francaises,
    construire_correspondances,
    ajouter_departements
)

# 2. Analyse de la régularité des TGV

## 2.1. Récupération et préparation des données SNCF


Les données ferroviaires proviennent du jeu de données *Régularité mensuelle TGV*, disponible sur la plateforme Open Data de la SNCF. Elles renseignent, pour chaque liaison et chaque mois, le nombre de trains prévus, les annulations et différents indicateurs de retard.

Afin d'automatiser la collecte et d'assurer la reproductibilité du projet, les données sont récupérées directement depuis l'API SNCF au moyen de requêtes HTTP, sans téléchargement manuel des fichiers. La période étudiée s'étend de **janvier 2018 à septembre 2026**.


In [4]:
# Récupération des données SNCF
df_sncf_brut = charger_sncf()

print("Dimensions de la base initiale :", df_sncf_brut.shape)
display(df_sncf_brut.head())

Dimensions de la base initiale : (12907, 26)


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation,commentaire_annulation,nb_train_depart_retard,retard_moyen_depart,...,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15,nb_train_retard_sup_30,nb_train_retard_sup_60,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


### Exploration des gares

Avant de définir le périmètre géographique de notre étude, nous examinons les gares présentes dans la base initiale.

Cette exploration permet d'identifier les éventuelles gares étrangères, ainsi que les valeurs manquantes ou les appellations particulières qui pourraient nécessiter un traitement.

In [8]:
# Exploration des gares
gares = lister_gares(df_sncf_brut)

print("Nombre de gares distinctes :", len(gares))
print("\nListe des gares :")
print(gares)

# Analyse des valeurs manquantes
valeurs_manquantes = pd.DataFrame({
    "Nombre de valeurs manquantes": df_sncf_brut.isna().sum(),
    "Pourcentage (%)": df_sncf_brut.isna().mean() * 100
})

display(
    valeurs_manquantes
    .sort_values("Nombre de valeurs manquantes", ascending=False)
    .round(2)
)

Nombre de gares distinctes : 59

Liste des gares :
['AIX EN PROVENCE TGV', 'ANGERS SAINT LAUD', 'ANGOULEME', 'ANNECY', 'ARRAS', 'AVIGNON TGV', 'BARCELONA', 'BELLEGARDE (AIN)', 'BESANCON FRANCHE COMTE TGV', 'BORDEAUX ST JEAN', 'BREST', 'CHAMBERY CHALLES LES EAUX', 'DIJON VILLE', 'DOUAI', 'DUNKERQUE', 'FRANCFORT', 'GENEVE', 'GRENOBLE', 'ITALIE', 'LA ROCHELLE VILLE', 'LAUSANNE', 'LAVAL', 'LE CREUSOT MONTCEAU MONTCHANIN', 'LE MANS', 'LILLE', 'LYON PART DIEU', 'MACON LOCHE', 'MADRID', 'MARNE LA VALLEE', 'MARSEILLE ST CHARLES', 'METZ', 'MONTPELLIER', 'MULHOUSE VILLE', 'NANCY', 'NANTES', 'NICE VILLE', 'NIMES', 'PARIS EST', 'PARIS LYON', 'PARIS MONTPARNASSE', 'PARIS NORD', 'PARIS VAUGIRARD', 'PERPIGNAN', 'POITIERS', 'QUIMPER', 'REIMS', 'RENNES', 'SAINT ETIENNE CHATEAUCREUX', 'ST MALO', 'ST PIERRE DES CORPS', 'STRASBOURG', 'STUTTGART', 'TOULON', 'TOULOUSE MATABIAU', 'TOURCOING', 'TOURS', 'VALENCE ALIXAN TGV', 'VANNES', 'ZURICH']


,Nombre de valeurs manquantes,Pourcentage (%)
commentaire_annulation,12907,100.00
commentaire_retards_depart,12907,100.00
commentaires_retard_arrivee,12209,94.59
gare_depart,0,0.00
service,0,0.00
gare_arrivee,0,0.00
nb_train_prevu,0,0.00
nb_annulation,0,0.00
duree_moyenne,0,0.00
date,0,0.00


L'exploration de la base révèle **59 appellations de gares distinctes**, dont huit correspondent à des destinations étrangères. Concernant les valeurs manquantes, seules trois variables de commentaires sont concernées. Ces variables n'étant pas utilisées dans notre analyse, aucune imputation n'est nécessaire.

Afin de conserver un périmètre cohérent avec les données météorologiques françaises, nous excluons les liaisons dont la gare de départ ou d'arrivée se situe à l'étranger.

In [9]:
# Conservation des liaisons françaises
df_sncf_fr = filtrer_liaisons_francaises(df_sncf_brut)

print("Observations initiales :", len(df_sncf_brut))
print("Observations conservées :", len(df_sncf_fr))

Observations initiales : 12907
Observations conservées : 11357
